In [1]:
# importing libraries

import numpy as np
import pandas as pd
import joblib
import duckdb
from collections import Counter, defaultdict

# tensorflow
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.losses import BinaryCrossentropy

%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
# from google paper, we have learnt that neural networks are good in generalization
# we have to suport the model by providing cross transformations that 
# might help the model learn user specific details
# hence wide model will only be used to learn cross transformations 
# and rest all the dense/cat features will be learnt by deep network 

train_df = pd.read_pickle('data/intermediate_files/ranker_train_df.pkl')
val_df = pd.read_pickle('data/intermediate_files/ranker_validation_df.pkl')
test_df = pd.read_pickle('data/intermediate_files/ranker_test_df.pkl')

In [3]:
print(train_df.shape, val_df.shape, test_df.shape)

(3422822, 33) (8979560, 33) (9340931, 33)


### Preprocessing

##### Deterministic preprocessing

In [4]:
# creating string columns to later create crosses on them

def rank_to_bucket(rank):

    if pd.isna(rank):
        return 'unknown'
    elif rank <= 10:
        return "1_10"
    elif rank <= 50:
        return "11_50"
    elif rank <= 100:
        return "51_100"
    elif rank <= 250:
        return "101_250"
    elif rank <= 500:
        return "251_500"
    else:
        return 'unknown'
    
def add_rank_bucket_cols(df, rank_cols):

    df = df.copy()

    for col in rank_cols:
        if col in df.columns:
            df[f'{col}_bucket'] = df[col].apply(rank_to_bucket).astype(str)

    return df

In [5]:
# wide features from the new columns

def add_wide_crosses(df, flag_cols, str_cols):

    """
    flag cols - the columns which are 1/0 in nature
    str_cols - columns which are have string values 
    """

    # create combination of columns for crosses

    df = df.copy()

    # ensure all columns are str for str_cols
    for col in str_cols:
        df[col] = df[col].fillna('unknown').astype(str)

    # ensure all columns are 1/0 but still string in flag_cols
    for col in flag_cols:
        df[col] = df[col].fillna(0).astype(int)
        df[f'{col}_str'] = df[col].astype(str)

    # Useful wide crosses
    df["cross_userhist_x_cgrank"] = df["user_history_bucket"] + "__" + df["cg_rank_bucket"]

    
    df["cross_cgrank_x_popularity"] = df["cg_rank_bucket"] + "__" + df["popularity_bucket"]

    df["cross_cgrank_x_generality"] = df["cg_rank_bucket"] + "__" + df["generality_bucket"]


    df["cross_category_x_popularity"] = df["categoryid"] + "__" + df["popularity_bucket"]

    df["cross_category_x_generality"] = df["categoryid"] + "__" + df["generality_bucket"]

    df["cross_parent_catid_x_popularity"] = df["parent_category_id"] + "__" + df["popularity_bucket"]

    df["cross_parent_catid_x_generality"] = df["parent_category_id"] + "__" + df["generality_bucket"]
    

    df["cross_userhist_x_cointeraction"] = df["user_history_bucket"] + "__" + df["from_user_co_interaction_str"]
        
    df["cross_userhist_x_usercategory"] = df["user_history_bucket"] + "__" + df["from_user_category_str"]

    df["cross_userhist_x_usergenerality"] = df["user_history_bucket"] + "__" + df["from_user_generality_str"]

    df["source_combo"] = (
            df["from_user_co_interaction"].astype(str) + "_" +
            df["from_user_category"].astype(str) + "_" +
            df["from_user_generality"].astype(str)
        )
    
    df["cross_sourcecombo_x_cgrank"] =  df["source_combo"] + "__" + df["cg_rank_bucket"]

    return df

In [6]:
def preprocess_pre_sample(df):
    tp_df = df.copy()

    # Time features
    tp_df["query_time"] = pd.to_datetime(tp_df["query_time"])
    tp_df["day"] = tp_df["query_time"].dt.day_name().astype(str)
    tp_df["hour_of_day"] = tp_df["query_time"].dt.hour.astype(str)

    # Log features for skewed numeric columns
    for col in ["past_view_count", "past_atc_count", "past_order_count", "cg_score_raw"]:
        tp_df[col] = tp_df[col].fillna(0)
        tp_df[f"log_{col}"] = np.log1p(tp_df[col].astype(float))

    # Basic null handling
    tp_df["cg_score_norm"] = tp_df["cg_score_norm"].fillna(0)
    tp_df["user_history_bucket"] = tp_df["user_history_bucket"].fillna("unknown").astype(str)

    # Binary columns
    binary_base_cols = [
        "candidate_previously_purchased", "candidate_in_last_k_items",
        "from_user_co_interaction", "from_user_category", "from_user_generality"
    ]
    for col in binary_base_cols:
        if col in tp_df.columns:
            tp_df[col] = tp_df[col].fillna(0).astype("int8")

    # Rank bucket flags; NaN rank becomes 0 for every bucket
    rank_cols = ["cg_rank", "user_co_interaction_rank", "user_category_rank", "user_generality_rank"]
    bucket_defs = {
        "1_10": (1, 10),
        "11_50": (11, 50),
        "51_100": (51, 100),
        "101_250": (101, 250),
        "251_500": (251, 500),
    }

    for col in rank_cols:
        if col not in tp_df.columns:
            continue
        for bucket_name, (lo, hi) in bucket_defs.items():
            tp_df[f"{col}_bucket_{bucket_name}"] = tp_df[col].between(lo, hi).fillna(False).astype("int8")

    # create features for crosses
    tp_df = add_rank_bucket_cols(tp_df, rank_cols)

    # create cross features
    flag_cols = [col for col in tp_df.columns if "_bucket_" in col] + [col for col in binary_base_cols]
    str_cols = [col for col in tp_df.columns if "_bucket" in col and col not in flag_cols] + ['day', 'user_history_bucket', 'hour_of_day', 'categoryid', 'parent_category_id']

    tp_df = add_wide_crosses(tp_df, flag_cols, str_cols)

    return tp_df

##### Negative sampling

In [7]:

def sample_negatives_by_query(df, label_col="label_engagement", query_col="global_query_id", neg_per_pos=10, random_state=42):
    sampled_groups = []
    rng = np.random.default_rng(random_state)

    for _, group in df.groupby(query_col):
        pos = group[group[label_col] == 1]
        neg = group[group[label_col] == 0]

        # Skip all-negative queries for first ranker training
        if len(pos) == 0:
            continue

        n_neg = min(len(neg), len(pos) * neg_per_pos)

        if n_neg > 0:
            neg_sample = neg.sample(n=n_neg, random_state=int(rng.integers(0, 1_000_000)))
            sampled_groups.append(pd.concat([pos, neg_sample], axis=0))
        else:
            sampled_groups.append(pos)

    sampled_df = pd.concat(sampled_groups, ignore_index=True)
    sampled_df = sampled_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return sampled_df

##### feature cleaning & others

In [8]:
def clean_features(df, wide_cat_features,
                    deep_dense_features, deep_cat_features):
    df = df.copy()

    for col in deep_dense_features:
        df[col] = df[col].fillna(0).astype("float32")

    for col in set(wide_cat_features + deep_cat_features):
        df[col] = df[col].fillna('unknown').astype(str)

    return df

In [9]:
# sampling validation/test to make it smaller, might comment this later

def sample_query_groups(df, n_queries, query_col='global_query_id', random_state=42):

    query_ids = df[query_col].drop_duplicates()
    sample_ids = query_ids.sample(n=min(n_queries, len(query_ids)), random_state=random_state)

    return df[df[query_col].isin(sample_ids)].copy()

### Tensorflow dataset creation

In [10]:
def make_tf_dataset(df, wide_cat_features,
                    deep_dense_features, deep_cat_features, label_col, 
                    batch_size=512, shuffle=False, random_state=42):

    feature_dict = {}
    
    feature_dict['deep_dense_features'] = df[deep_dense_features].astype('float32').values
    
    # Separate categorical inputs
    for col in set(wide_cat_features + deep_cat_features):
        feature_dict[col] = df[col].astype(str).values
    
    labels = df[label_col].astype('float32').values

    # creates tensorflow training input
    ds = tf.data.Dataset.from_tensor_slices((feature_dict, labels))

    # shuffling the dataset using a rolling window of 1L rows at a time 
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(df), 1_00_000), seed=random_state)

    # prepares data in form of batches, and also prefetch(tf.data.AUTOTUNE) improves 
    # performance by preparing the next training batch while the current training is happening
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    return ds

In [11]:
def build_item_vocab(train_df, item_col="candidate_itemid", 
                     min_freq=2, max_items=50000):
    item_counts = train_df[item_col].astype(str).value_counts()

    # value counts result in an descending sorted output, so head(max_items) will return the top 50k items based on count
    item_vocab = (
        item_counts[item_counts >= min_freq]
        .head(max_items)
        .index
        .tolist()
    )

    return item_vocab


In [12]:
def build_lookup_layers(train_df, embedding_cat_cols):

    lookup_layers = {}
    item_vocab = None

    for col in embedding_cat_cols:
        if col == 'candidate_itemid':
            vocab = build_item_vocab(train_df, item_col=col)
            item_vocab = vocab.copy()
        else:
            vocab = train_df[col].astype(str).unique().tolist()

        lookup_layers[col] = layers.StringLookup(
            vocabulary=vocab, mask_token=None, 
            num_oov_indices=1, output_mode="int", name=f'{col}_lookup'
        )

    return lookup_layers, item_vocab

### Model Build

In [13]:
def one_hot_encoding_helper(input_tensor, lookup_layer, name_prefix):

    x = lookup_layer(input_tensor)

    x = layers.CategoryEncoding(
        num_tokens=lookup_layer.vocabulary_size(),
        output_mode='one_hot',
        name=f'{name_prefix}_one_hot')(x)
    
    return x

In [14]:
def embedding_dim_helper(col, vocab_size):

    # embedding size for later use
    if col == 'candidate_itemid':
        return 16
    elif col == 'categoryid':
        return 16
    elif col == 'parent_category_id':
        return 8
    elif col in ['hour_of_day', 'day', 'user_history_bucket', "popularity_bucket", "generality_bucket"]:
        return 4
    else:
        return min(16, max(4, int(np.sqrt(vocab_size))))


In [15]:
### Wide & Deep Ranker v1

def build_wide_n_deep_ranker_v1(deep_dense_ip_dim, deep_emb_cat_cols, 
                                wide_emb_cat_cols, lookup_layers):

    # input prep
    inputs = {}
    wide_parts = []
    deep_parts = []
    
    # steps
    # dense features - only deep
    # prepare input for dense/binary features
    # normalize using batch norm

    # cat cols
    # wide
    # prepare input with shape=()
    # get lookup value
    # one hot encoding using CategoryEncoding
    
    # deep
    # prepare input with shape=()
    # get lookup value and vocab size and embedding size
    # lookup(input) 
    # embedding using layers.Embedding
    # flatten using layers.Flatten

    # deep
    deep_dense_input = layers.Input(shape=(deep_dense_ip_dim,), name='deep_dense_features')
    x_deep_dense = layers.BatchNormalization(name='dense_input_norm')(deep_dense_input) 
        
    inputs['deep_dense_features'] = deep_dense_input
    deep_parts.append(x_deep_dense)

    # wide embedding inputs
    for col in wide_emb_cat_cols:
        cat_input = layers.Input(shape=(), dtype=tf.string, name=col)
        inputs[col] = cat_input

        one_hot = one_hot_encoding_helper(cat_input, lookup_layers[col], 
                                          f'wide_{col}')
        
        wide_parts.append(one_hot)

    # deep embedding inputs
    for col in deep_emb_cat_cols:
        cat_input = layers.Input(shape=(), dtype=tf.string, name=col)
        inputs[col] = cat_input

        lookup = lookup_layers[col]
        vocab_size = lookup.vocabulary_size()
        
        # embedding size 
        emb_dim = embedding_dim_helper(col, vocab_size)

        x_deep_cat = lookup(cat_input)

        x_deep_cat = layers.Embedding(input_dim=vocab_size,
            output_dim=emb_dim,
            name=f"{col}_embedding")(x_deep_cat)

        x_deep_cat = layers.Flatten(name=f'{col}_flatten')(x_deep_cat)
        deep_parts.append(x_deep_cat)


    # concatenate deep dense features and embeddings
    x_deep = layers.Concatenate(name='deep_concat')(deep_parts)

    # concatenate wide features
    x_wide = layers.Concatenate(name='wide_concat')(wide_parts)    

    # model prep - deep
    # hidden layers
    x_deep = layers.Dense(units=128, activation='relu', name='deep_layer1')(x_deep)

    x_deep = layers.Dense(units=64, activation='relu', name='deep_layer2')(x_deep)
    
    x_deep = layers.Dense(units=32, activation='relu', name='deep_layer3')(x_deep)
    # output layer - deep
    deep_op_logits = layers.Dense(units=1, activation='linear', name="deep_score")(x_deep)

    # model prep - wide
    wide_op_logits = layers.Dense(units=1, activation='linear', name="wide_score")(x_wide)

    # combining all to form the model for joint training
    final_op_logits = layers.Add(name='wide_n_deep_score')([wide_op_logits, deep_op_logits])

    # creating the model
    model = Model(inputs=inputs, outputs=final_op_logits)    

    return model


### Train and Validation Data build

In [16]:
# train DS

neg_per_pos = 10
random_state = 42
label_col = 'label_engagement'

# Deterministic features first
train_pre = preprocess_pre_sample(train_df)

# Negative sampling only on train
train_sampled = sample_negatives_by_query(
    train_pre,
    label_col=label_col,
    query_col="global_query_id",
    neg_per_pos=neg_per_pos,
    random_state=random_state
)

In [17]:
## creating column name lists

continuous_cols = ["log_past_view_count", "log_past_atc_count", "log_past_order_count", "log_cg_score_raw", "cg_score_norm",]

binary_cols = ["candidate_previously_purchased", "candidate_in_last_k_items", "from_user_co_interaction", "from_user_category",
    "from_user_generality"]

rank_bucket_cols = [col for col in train_sampled.columns if "_bucket_" in col]

deep_cat_cols = ["user_history_bucket", "day", "hour_of_day",
                    'categoryid', 'parent_category_id', 'generality_bucket', 'popularity_bucket', "source_combo"]

wide_cat_cols = [col for col in train_sampled.columns if "cross" in col]

deep_dense_cols = continuous_cols+binary_cols+rank_bucket_cols

In [18]:
# getting all the relevant columns - sanity
wide_cat_features = [c for c in wide_cat_cols if c in train_sampled.columns]
deep_cat_features = [c for c in deep_cat_cols if c in train_sampled.columns]

deep_dense_features = [c for c in deep_dense_cols if c in train_sampled.columns]

# final cleaning
train_model_df = clean_features(train_sampled, wide_cat_features,
                    deep_dense_features, deep_cat_features)

# creating tenssorflow datasets
train_ds = make_tf_dataset(train_model_df, wide_cat_features,
                    deep_dense_features, deep_cat_features, label_col, shuffle=True)

In [19]:
# Val DS

# sample queries from validation df
val_small = sample_query_groups(val_df, 10000, query_col='global_query_id', random_state=42)

# Deterministic features first
val_pre = preprocess_pre_sample(val_small)

# final cleaning
val_model_df = clean_features(val_pre, wide_cat_features,
                    deep_dense_features, deep_cat_features)

# creating tenssorflow datasets
val_ds = make_tf_dataset(val_model_df, wide_cat_features,
                    deep_dense_features, deep_cat_features, label_col, shuffle=False)

In [20]:
# Sanity checks:
print("Train shape:", train_model_df.shape)
print("Positive rate:", train_model_df[label_col].mean())
print("Dense feature count:", len(deep_dense_features))
print("categorical cols - wide:", wide_cat_features)
print("Embedding categorical cols - deep:", deep_cat_features)
print("Nulls in dense features:", train_model_df[deep_dense_features].isna().sum().sum())


Train shape: (60654, 100)
Positive rate: 0.09090909090909091
Dense feature count: 50
categorical cols - wide: ['cross_userhist_x_cgrank', 'cross_cgrank_x_popularity', 'cross_cgrank_x_generality', 'cross_category_x_popularity', 'cross_category_x_generality', 'cross_parent_catid_x_popularity', 'cross_parent_catid_x_generality', 'cross_userhist_x_cointeraction', 'cross_userhist_x_usercategory', 'cross_userhist_x_usergenerality', 'cross_sourcecombo_x_cgrank']
Embedding categorical cols - deep: ['user_history_bucket', 'day', 'hour_of_day', 'categoryid', 'parent_category_id', 'generality_bucket', 'popularity_bucket', 'source_combo']
Nulls in dense features: 0


### Model Initialisation

In [21]:
# creating lookup
lookup_layers, item_vocab = build_lookup_layers(train_model_df, wide_cat_features+deep_cat_features)

In [22]:
# building the model

w_n_d_model_v1 = build_wide_n_deep_ranker_v1(deep_dense_ip_dim=len(deep_dense_features), 
                                           deep_emb_cat_cols=deep_cat_features, 
                                            wide_emb_cat_cols=wide_cat_features, 
                                            lookup_layers=lookup_layers)


w_n_d_model_v1.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(from_logits=True), 
    metrics=[
        tf.keras.metrics.AUC(name='roc_auc', from_logits=True),
        tf.keras.metrics.AUC(curve='PR', name='pr_auc', from_logits=True)
    ]
    )

w_n_d_model_v1.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_history_bucket │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day (InputLayer)    │ (None)            │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hour_of_day         │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ categoryid          │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ parent_category_id  │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ generality_bucket   │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ popularity_bucket   │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ source_combo        │ (None)            │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_history_bucke… │ (None)            │          0 │ user_history_buc… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_lookup          │ (None)            │          0 │ day[0][0]         │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ hour_of_day_lookup  │ (None)            │          0 │ hour_of_day[0][0] │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ categoryid_lookup   │ (None)            │          0 │ categoryid[0][0]  │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ parent_category_id… │ (None)            │          0 │ parent_category_… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ generality_bucket_… │ (None)            │          0 │ generality_bucke… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ popularity_bucket_… │ (None)            │          0 │ popularity_bucke… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ source_combo_lookup │ (None)            │          0 │ source_combo[0][… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ deep_dense_features │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                 

 Total params: 45,726 (178.62 KB)

 Trainable params: 45,626 (178.23 KB)

 Non-trainable params: 100 (400.00 B)

In [23]:
trial_model_history = w_n_d_model_v1.fit(train_ds, epochs=10, validation_data=val_ds)

Epoch 1/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 66s 509ms/step - loss: 0.1834 - pr_auc: 0.6663 - roc_auc: 0.8959 - val_loss: 0.0998 - val_pr_auc: 0.0114 - val_roc_auc: 0.9217
Epoch 2/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 449ms/step - loss: 0.1373 - pr_auc: 0.7813 - roc_auc: 0.9367 - val_loss: 0.0608 - val_pr_auc: 0.0120 - val_roc_auc: 0.9373
Epoch 3/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 55s 465ms/step - loss: 0.1324 - pr_auc: 0.7938 - roc_auc: 0.9438 - val_loss: 0.0498 - val_pr_auc: 0.0121 - val_roc_auc: 0.9357
Epoch 4/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 447ms/step - loss: 0.1297 - pr_auc: 0.7987 - roc_auc: 0.9473 - val_loss: 0.0467 - val_pr_auc: 0.0121 - val_roc_auc: 0.9363
Epoch 5/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 447ms/step - loss: 0.1271 - pr_auc: 0.8060 - roc_auc: 0.9503 - val_loss: 0.0530 - val_pr_auc: 0.0114 - val_roc_auc: 0.9340
Epoch 6/10
119/119 ━━━━━━━━━━━━━━━━━━━━ 53s 446ms/step - loss: 0.1252 - pr_auc: 0.8096 - roc_auc: 0.9523 - val_loss: 0.0442 - val_pr_auc: 0.0123 - val_roc_auc: 0.934

### Predictions

In [24]:
def predict_scores_in_chunks(df, model, wide_cat_features,
                             deep_dense_features, deep_cat_features,
                            label_col='label_engagement', chunk_size=300_000):
    
    scores = np.empty(len(df), dtype=np.float32)

    for start in range(0, len(df), chunk_size):
        end = min(start + chunk_size, len(df))

        chunk = df.iloc[start:end].copy()
        chunk = preprocess_pre_sample(chunk)
        chunk = clean_features(chunk, wide_cat_features,
                             deep_dense_features, deep_cat_features)

        ds = make_tf_dataset(chunk, wide_cat_features,
                             deep_dense_features, deep_cat_features, label_col, shuffle=False)

        raw_preds = model.predict(ds, verbose=0)

        preds = tf.nn.sigmoid(raw_preds).numpy().reshape(-1)

        scores[start:end] = preds.astype(np.float32)
        print(f"Scored rows {start:,} to {end:,}")

    return scores

### Evaluation and Lift

In [25]:
def evaluate_ranking(df, score_col, label_col="label_engagement", query_col="global_query_id",
                     ks=[10, 20, 50], mode="reranker_only"):
    eval_df = df.copy()

    if mode == "reranker_only":
        
        q_pos = eval_df.groupby(query_col)[label_col].sum()
        keep_qids = q_pos[q_pos > 0].index
        eval_df = eval_df[eval_df[query_col].isin(keep_qids)].copy()

    eval_df = eval_df.sort_values([query_col, score_col, "cg_rank"], ascending=[True, False, True])

    rows = []
    n_queries = eval_df[query_col].nunique()

    for k in ks:
        hit_list, recall_list, ndcg_list, mrr_list = [], [], [], []

        for _, g in eval_df.groupby(query_col, sort=False):
            labels = g[label_col].to_numpy()
            total_pos = labels.sum()

            if total_pos == 0:
                hit_list.append(0.0)
                recall_list.append(0.0)
                ndcg_list.append(0.0)
                mrr_list.append(0.0)
                continue

            topk = labels[:k]
            hits = topk.sum()

            hit = 1.0 if hits > 0 else 0.0
            recall = hits / total_pos

            discounts = 1.0 / np.log2(np.arange(2, len(topk) + 2))
            dcg = np.sum(topk * discounts)

            ideal_hits = int(min(total_pos, k))
            ideal_discounts = 1.0 / np.log2(np.arange(2, ideal_hits + 2))
            idcg = np.sum(ideal_discounts)
            ndcg = dcg / idcg if idcg > 0 else 0.0

            pos_idx = np.where(topk == 1)[0]
            mrr = 1.0 / (pos_idx[0] + 1) if len(pos_idx) > 0 else 0.0

            hit_list.append(hit)
            recall_list.append(recall)
            ndcg_list.append(ndcg)
            mrr_list.append(mrr)

        rows.append({
            "score_col": score_col,
            "mode": mode,
            "K": k,
            "n_queries": n_queries,
            "HitRate": np.mean(hit_list),
            "Recall": np.mean(recall_list),
            "NDCG": np.mean(ndcg_list),
            "MRR": np.mean(mrr_list),
        })

    return pd.DataFrame(rows)

In [26]:
def add_lift(compare_df, baseline_score_col="cg_baseline_score", model_score_col="lr_score"):
    base = compare_df[compare_df["score_col"] == baseline_score_col].copy()
    model = compare_df[compare_df["score_col"] == model_score_col].copy()

    merged = model.merge(
        base,
        on=["mode", "K"],
        suffixes=("_model", "_baseline")
    )

    for metric in ["HitRate", "Recall", "NDCG", "MRR"]:
        merged[f"{metric}_lift_abs"] = merged[f"{metric}_model"] - merged[f"{metric}_baseline"]
        merged[f"{metric}_lift_pct"] = np.where(
            merged[f"{metric}_baseline"] > 0,
            100 * merged[f"{metric}_lift_abs"] / merged[f"{metric}_baseline"],
            np.nan
        )

    return merged

### Evaluation Queries

In [27]:
# validation data
# val_scored = val_small[["global_query_id", "candidate_itemid", "cg_rank", "label_engagement"]].copy()

val_scored = val_df[["global_query_id", "candidate_itemid", "cg_rank", "label_engagement"]].copy()
val_scored['WnD_v1_score'] = predict_scores_in_chunks(val_df, w_n_d_model_v1, wide_cat_features, deep_dense_features, deep_cat_features)

val_scored["cg_baseline_score"] = -val_scored["cg_rank"]

ks = [10, 20, 50, 100]

# reranker only
val_cg_rerank = evaluate_ranking(
    val_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

val_dnn_rerank = evaluate_ranking(
    val_scored,
    score_col="WnD_v1_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

# all queries
val_cg_all = evaluate_ranking(
    val_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

val_dnn_all = evaluate_ranking(
    val_scored,
    score_col="WnD_v1_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

# val dfs

val_compare_rerank = pd.concat([val_cg_rerank, val_dnn_rerank], ignore_index=True)
val_compare_all = pd.concat([val_cg_all, val_dnn_all], ignore_index=True)

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,300,000
Scored rows 3,300,000 to 3,600,000
Scored rows 3,600,000 to 3,900,000
Scored rows 3,900,000 to 4,200,000
Scored rows 4,200,000 to 4,500,000
Scored rows 4,500,000 to 4,800,000
Scored rows 4,800,000 to 5,100,000
Scored rows 5,100,000 to 5,400,000
Scored rows 5,400,000 to 5,700,000
Scored rows 5,700,000 to 6,000,000
Scored rows 6,000,000 to 6,300,000
Scored rows 6,300,000 to 6,600,000
Scored rows 6,600,000 to 6,900,000
Scored rows 6,900,000 to 7,200,000
Scored rows 7,200,000 to 7,500,000
Scored rows 7,500,000 to 7,800,000
Scored rows 7,800,000 to 8,100,000
Scored rows 8,100,000 to 8,400,000
Scored rows 8,400,000 to 8,700,000
Score

In [28]:
# test data - done on whole test data, can be sampled as well if needed.
# test_small = sample_query_groups(test_df, 10000, query_col='global_query_id', random_state=42)

test_scored = test_df[["global_query_id", "candidate_itemid", "cg_rank", "label_engagement"]].copy()
test_scored['WnD_v1_score'] = predict_scores_in_chunks(test_df, w_n_d_model_v1, wide_cat_features, deep_dense_features, deep_cat_features)

test_scored["cg_baseline_score"] = -test_scored["cg_rank"]

ks = [10, 20, 50, 100]

# reranker only
test_cg_rerank = evaluate_ranking(
    test_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

test_dnn_rerank = evaluate_ranking(
    test_scored,
    score_col="WnD_v1_score",
    label_col="label_engagement",
    ks=ks,
    mode="reranker_only"
)

# all queries
test_cg_all = evaluate_ranking(
    test_scored,
    score_col="cg_baseline_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

test_dnn_all = evaluate_ranking(
    test_scored,
    score_col="WnD_v1_score",
    label_col="label_engagement",
    ks=ks,
    mode="all_queries"
)

# test dfs

test_compare_rerank = pd.concat([test_cg_rerank, test_dnn_rerank], ignore_index=True)
test_compare_all = pd.concat([test_cg_all, test_dnn_all], ignore_index=True)

Scored rows 0 to 300,000
Scored rows 300,000 to 600,000
Scored rows 600,000 to 900,000
Scored rows 900,000 to 1,200,000
Scored rows 1,200,000 to 1,500,000
Scored rows 1,500,000 to 1,800,000
Scored rows 1,800,000 to 2,100,000
Scored rows 2,100,000 to 2,400,000
Scored rows 2,400,000 to 2,700,000
Scored rows 2,700,000 to 3,000,000
Scored rows 3,000,000 to 3,300,000
Scored rows 3,300,000 to 3,600,000
Scored rows 3,600,000 to 3,900,000
Scored rows 3,900,000 to 4,200,000
Scored rows 4,200,000 to 4,500,000
Scored rows 4,500,000 to 4,800,000
Scored rows 4,800,000 to 5,100,000
Scored rows 5,100,000 to 5,400,000
Scored rows 5,400,000 to 5,700,000
Scored rows 5,700,000 to 6,000,000
Scored rows 6,000,000 to 6,300,000
Scored rows 6,300,000 to 6,600,000
Scored rows 6,600,000 to 6,900,000
Scored rows 6,900,000 to 7,200,000
Scored rows 7,200,000 to 7,500,000
Scored rows 7,500,000 to 7,800,000
Scored rows 7,800,000 to 8,100,000
Scored rows 8,100,000 to 8,400,000
Scored rows 8,400,000 to 8,700,000
Score

In [29]:
val_lift_rerank = add_lift(val_compare_rerank, baseline_score_col="cg_baseline_score", model_score_col="WnD_v1_score")
val_lift_all = add_lift(val_compare_all, baseline_score_col="cg_baseline_score", model_score_col="WnD_v1_score")

val_lift_rerank[[
    "mode", "K",
    "NDCG_baseline", "NDCG_model", "NDCG_lift_abs", "NDCG_lift_pct",
    "HitRate_baseline", "HitRate_model", "HitRate_lift_abs", "HitRate_lift_pct"
]]

,mode,K,NDCG_baseline,NDCG_model,NDCG_lift_abs,NDCG_lift_pct,HitRate_baseline,HitRate_model,HitRate_lift_abs,HitRate_lift_pct
0,reranker_only,10,0.179537,0.549264,0.369726,205.932583,0.344123,0.713799,0.369676,107.425743
1,reranker_only,20,0.217969,0.569304,0.351334,161.185246,0.490630,0.780239,0.289608,59.027778
2,reranker_only,50,0.255804,0.587327,0.331523,129.600683,0.676320,0.853492,0.177172,26.196474
3,reranker_only,100,0.286788,0.596447,0.309659,107.974792,0.843271,0.899489,0.056218,6.666667


In [30]:
test_lift_rerank = add_lift(test_compare_rerank, baseline_score_col="cg_baseline_score", model_score_col="WnD_v1_score")
test_lift_all = add_lift(test_compare_all, baseline_score_col="cg_baseline_score", model_score_col="WnD_v1_score")

test_lift_rerank[[
    "mode", "K",
    "NDCG_baseline", "NDCG_model", "NDCG_lift_abs", "NDCG_lift_pct",
    "HitRate_baseline", "HitRate_model", "HitRate_lift_abs", "HitRate_lift_pct"
]]

,mode,K,NDCG_baseline,NDCG_model,NDCG_lift_abs,NDCG_lift_pct,HitRate_baseline,HitRate_model,HitRate_lift_abs,HitRate_lift_pct
0,reranker_only,10,0.182651,0.515037,0.332386,181.979096,0.328,0.692,0.364,110.975610
1,reranker_only,20,0.211975,0.536004,0.324029,152.861917,0.440,0.760,0.320,72.727273
2,reranker_only,50,0.257558,0.554393,0.296835,115.250044,0.670,0.840,0.170,25.373134
3,reranker_only,100,0.285076,0.567590,0.282514,99.101101,0.814,0.908,0.094,11.547912


### Storing a common df with all scores for comparison later

In [31]:
# # the scored tables might already exist in data/final_files/,
# # because i created them when working with neural networks
# #  so i will read then first and then predict on my data

# # read the csvs
# train_scored2 = pd.read_csv('data/final_files/train_scored.csv')
# val_scored2 = pd.read_csv('data/final_files/val_scored.csv')
# test_scored2 = pd.read_csv('data/final_files/test_scored.csv')

# assert len(train_scored2) == len(train_df)
# assert len(val_scored2) == len(val_df)
# assert len(test_scored2) == len(test_df)

# # producing the model results
# train_scored2["WnD_v1_score"] = predict_scores_in_chunks(train_df, w_n_d_model_v1, wide_cat_features, deep_dense_features, deep_cat_features)

# val_scored2["WnD_v1_score"] = val_scored['WnD_v1_score']

# test_scored2["WnD_v1_score"] = test_scored['WnD_v1_score']

In [32]:
# ### Saving the models

# w_n_d_model_v1.save("models/deep_model/wide_n_deep_model_v1.keras")

# # saving the scored dfs
# train_scored2.to_csv("data/final_files/train_scored.csv", index=False)
# val_scored2.to_csv("data/final_files/val_scored.csv", index=False)
# test_scored2.to_csv("data/final_files/test_scored.csv", index=False)

# # exporting test results
# test_compare_rerank.to_csv("reports/wide_n_deep_model/test_reranker_only_metrics_WnD.csv", index=False)
# test_compare_all.to_csv("reports/wide_n_deep_model/test_all_query_metrics_WnD.csv", index=False)